# Exercise 2: Particle Filter for Robot Localization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy.linalg

np.random.seed(123456)

# Global constants
dt = 0.1 # discretization timestep
Q = (dt**2 * 0.1) * np.eye(3) # process noise
R = 0.25 * np.eye(8) # observation noise
R_inv = np.linalg.inv(R) # precompute meas. noise covariance inverse
m = np.array([0, 0, 2, 8, 8, 2, 10, 10]) # ground-truth landmark positions.

### Exercise 2.1: Dynamics and Measurement Equations
Consider the discrete-time robot dynamics model:
\begin{equation*}
\begin{split}
x_{t+1} &= x_{t} + V_{t} \cos(\theta_{t}) \Delta t, \\
y_{t+1} &= y_{t} + V_{t} \sin(\theta_{t}) \Delta t \\
\theta_{t+1} &= \theta_{t} + \omega_t \Delta t,
\end{split}
\end{equation*}
where $(x,y)$ is the robot position, $\theta$ is the heading, $\Delta t$ is the discretization timestep, and $(V,\omega)$ are the speed and angular velocity commands.

In this problem, we assume the global ground truth positions of the four landmarks are known. 
The landmarks are stationary objects in the environment, and their combined state vector is:
\begin{equation*}
\begin{split}
\textbf{m} = \begin{bmatrix}
m_{1,x} & m_{1,y} & m_{2,x} & m_{2,y} & m_{3,x} & m_{3,y} & m_{4,x} & m_{4,y}
\end{bmatrix}^\top.
\end{split}
\end{equation*}
As the robot navigates through its environment, it receives noisy measurements of the positions of four landmarks in the environment relative to the robot's current pose. 
The measurement for landmark $i$ is the relative position with the measurement model:
\begin{equation*}
\begin{split}
\textbf{z}_t^{i} = 
\begin{bmatrix}
    \cos(\theta_{t}) & \sin(\theta_{t}) \\
    -\sin(\theta_{t}) & \cos(\theta_{t})
\end{bmatrix}
\Big(\begin{bmatrix}
    m_{i,x} \\ m_{i,y}
\end{bmatrix} - \begin{bmatrix}
    x_t \\ y_t
\end{bmatrix}\Big).
\end{split}
\end{equation*}
The full measurement vector of all landmarks is:
\begin{equation*}
\begin{split}
\textbf{z}_t = \begin{bmatrix} {\textbf{z}_t^{1}}^\top,  {\textbf{z}_t^{2}}^\top, {\textbf{z}_t^{3}}^\top, {\textbf{z}_t^{4}}^\top \end{bmatrix}^\top.
\end{split}
\end{equation*}

Implement these models in the functions `robot_dynamics` and `robot_measurement`.

NOTE: if you have already completed Exercise 1: Extended Kalman Filter for Robot Localization, you may reuse your solution for this part.

In [ ]:
def robot_dynamics(x: np.ndarray, u: np.ndarray) -> np.ndarray:
    """
    Robot dynamics model.

    Args:
        x: robot state, [x, y, θ]
        u: robot control vector [v, omega] where v is the
           linear velocity and omega is the angular velocity

    Returns:
        Expected next robot state.
    """
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######


def robot_measurement(x: np.ndarray, m: np.ndarray) -> np.ndarray:
    """
    Robot measurement model that measures the relative position of the landmarks.

    Args:
        x: robot state, [px, py, θ]
        m: landmark positions, [m_1,x, m_1,y, ..., m_4,y], size (8,)
    
    Returns:
        Expected measurement from the given robot pose.
    """
    px = x[0]
    py = x[1]
    th = x[2]
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######

### Exercise 2.2: Particle Filter Implementation
Implement the function `particle_filter_update` to implement the particle filter update.

In [ ]:
def gaussian_pdf(x: np.ndarray, mean: np.ndarray, cov_inv: np.ndarray) -> np.ndarray:
    """
    Helper function to compute (unnormalized) likelihood of a point x
    for a Gaussian distribution with given mean, and *inverse* covariance cov_inv.

    Args:
        x: value to get pdf value at, shape (n,)
        mean: mean of the Gaussian, shape (n,)
        cov_inv: inverse covariance matrix, shape (n,n)

    Returns:
        Gaussian pdf evaluated at x.
    """
    # note: don't compute the normalization factor because it doesn't matter.
    return np.exp(-0.5 * np.dot(x - mean, cov_inv @ (x - mean)))

In [ ]:
def particle_filter_update(
    particles: np.ndarray, 
    u: np.ndarray, 
    z: np.ndarray,
) -> np.ndarray:
    """
    Given the prior belief particle set, update the belief using the 
    given control input and measurements.

    Args:
        particles: array representing particles, shape (3, num_particles)
        u: control input, shape (2,)
        z: measurement array, shape (8,)

    Returns:
        Updated particle set, shape (3, num_particles)
    """
    # Sample particle noises
    n, num_particles = particles.shape
    W_particles = (np.linalg.cholesky(Q)
                    @ np.random.normal(size=(n, num_particles)))

    ##### YOUR CODE STARTS HERE #####
    # Hint: Use functions 'robot_dynamics()', 'robot_measurement()' and 'gaussian_pdf()'.
    # Hint: Do not forget to add the process noise to each particle, i.e., 'W_particles'.
    # Hint: Use function 'np.random.choice()' to resample.
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######
    return particles

### Exercise 2.3: Simulate PF Localization
Run the provided code below to simulate the robot PF localization given an open-loop control sequence and noisy measurements and dynamics.

In [ ]:
# Time horizon to simulate
tf = 15
time = np.arange(0, tf + dt, dt)

# Ground-truth states (simulated)
x = np.zeros((3, len(time)))
x[:, 0] = [1, 1, 0]

# Initial state and state covariance estimate
init_mu_std = 0.1
init_mean = [1, 1, 0] + np.random.multivariate_normal(np.zeros((3,)), init_mu_std * np.eye(3))
init_cov = np.eye(3)

# Containers for belief
T = len(time)
n = init_mean.shape[0]
mean_pf = np.zeros((n, T))
cov_pf = np.zeros((n, n, T))

# Containers for particles
num_particles = 1000
particles = np.zeros((n, num_particles))
updated_particles = np.zeros((n, num_particles))

# Sample particles
particles = (init_mean.reshape(n, 1) + scipy.linalg.sqrtm(init_cov)
                @ np.random.normal(size=(n, num_particles)))
for i in range(0, len(time) - 1):
    # Simulation
    # True robot commands
    v = 1
    omega = np.sin(time[i])
    u = np.array([v, omega])

    # True robot dynamics with noise
    w_noise = np.random.multivariate_normal(np.zeros((3,)), Q)
    x[:, i + 1] = robot_dynamics(x[:, i], u) + w_noise

    # True received measurement
    v_noise = np.random.multivariate_normal(np.zeros((8,)), R)
    z = robot_measurement(x[:, i + 1], m) + v_noise

    # Store the current belief's mean and covariance to 'mean_pf' and 'cov_pf'.
    mean_pf[:, i] = np.mean(particles, axis=1)
    cov_pf[:, :, i] = np.cov(particles)

    # Estimation.
    particles = particle_filter_update(particles, u, z)

# Store final belief
mean_pf[:, -1] = np.mean(particles, axis=1)
cov_pf[:, :, -1] = np.cov(particles)

Finally, run the code below to generate some plots to visualize the results.

In [ ]:
plt.figure(figsize=(18, 4))
titles = ['X Position', 'Y Position', 'Orientation']
ylabels = [r'$x_r$', r'$y_r$', r'$\theta_r$']
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.title(titles[i])
    plt.plot(time, x[i, :], linewidth=2, label='True')
    plt.plot(time, mean_pf[i, :], '.', markersize=3, label='PF')
    plt.xlabel('Time')
    plt.ylabel(ylabels[i])
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()

#### Plot Error Ellipses Along Trajectory

In [ ]:
def plot_error_ellipse(ax, mean, cov):
    # Calculate the error ellipse parameters
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))

    # Compute the radius of the ellipse to correspond to the desired confidence level
    chi2_val = 2.4477  # Corresponds to 95% conf. interval
    width, height = 2 * chi2_val * np.sqrt(eigenvalues)

    # Draw the ellipse
    ellipse = patches.Ellipse(mean, width, height, angle=angle, edgecolor='red', fc='None', lw=2)
    ax.add_patch(ellipse)

plt.figure(figsize=(10, 8))
plt.title('Estimated Robot Trajectory and Uncertainty')
plt.plot(x[0, :], x[1, :], linewidth=2, label='Ground truth robot')
plt.plot(mean_pf[0, :], mean_pf[1, :], color='orange', linewidth=2, label='PF estimate')
plt.plot(m.reshape((4,2))[:, 0], m.reshape((4,2))[:, 1], 'b.', markersize=12, label='Landmark')
for i in range(0, len(time), 10):
    plot_error_ellipse(plt.gca(), mean_pf[:2, i], cov_pf[:2, :2, i])
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True)
plt.show()